## Compile data from California WellStar program
* The "wells for 2020" report can be reached directly here: https://calgem-pid.conservation.ca.gov/pid/2020CaliforniaOilAndGasWells.csv)
* The "wells for 2022" report can be reached directly here: https://calgem-pid.conservation.ca.gov/pid/2022CaliforniaOilAndGasWells.csv
* This report lists fields (names and numbers) and corresponding wells, status, and operators

* The "well monthly production for 2020” report can be reached directly here: (https://calgem-pid.conservation.ca.gov/pid/2020CaliforniaOilAndGasWellMonthlyProduction.csv
* This report provides monthly oil, gas, and condensate production by field.

In [ ]:
# First to compile the monthly production data into an annual total for each well.
# Then, the API codes listed in the second (production data) report can be matched to the
# API numbers in the first report so that those production totals can be assigned to each
# operator and field name.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
goget_path = '/Users/jennymartos/Documents/GEM/GRFF/California/'
production_file = '2020CaliforniaOilAndGasWellMonthlyProduction.csv'
wells_file = '2020CaliforniaOilAndGasWells.csv'

In [ ]:
production_df = pd.read_csv(goget_path + production_file)
production_df = production_df.copy()

In [ ]:
# CHECK PRODUCTION TOTALS
total_gas_prod = production_df['GasProduced'].sum()
total_oilcondens_prod = production_df['OilorCondensateProduced'].sum()

print(f'Total gas production: {total_gas_prod}')
print(f'Total oil and condensate production: {total_oilcondens_prod}')

Total gas production: 295651129.63022107
Total oil and condensate production: 144536567.1321826


In [ ]:
# Production data clean up
production_df['OilorCondensateProduced'] = production_df['OilorCondensateProduced'].fillna('')
production_df['GasProduced'] = production_df['GasProduced'].fillna('')

# Sum production by API Number
list_of_col = ['OilorCondensateProduced','GasProduced']
production_df = production_df.groupby('APINumber')[list_of_col].sum().reset_index()

# production_df

In [ ]:
wells_df = pd.read_csv(goget_path + wells_file)
wells_df = wells_df.copy()

In [ ]:
#Add production columns to wells file, based on API number
wells_df = wells_df[['API','FieldName','OperatorName']].drop_duplicates(subset=['API'])
wells_df = wells_df.join(production_df.set_index('APINumber'), on='API')

In [ ]:
# Sum by Field Name & Operator
wells_df = wells_df.groupby(['FieldName','OperatorName'])[list_of_col].sum().reset_index()
wells_df

,FieldName,OperatorName,OilorCondensateProduced,GasProduced
0,Afton South Gas (ABD),Shell Western E&P Inc.,0.0,0.0
1,Afton Gas,California Resources Production Corporation,0.0,0.0
2,Afton Gas,Crain Orchards Inc.,0.0,0.0
3,Afton Gas,Donald C Slawson Expl Co. Inc,0.0,0.0
4,Afton Gas,Fountain Oil And Gas Co.,0.0,0.0
...,...,...,...,...
3769,Zamora North Gas (ABD),Chevron U.S.A. Inc.,0.0,0.0
3770,Zamora Gas (ABD),Arco Western Energy Co.,0.0,0.0
3771,Zamora Gas (ABD),Hartley Natural Resource Development,0.0,0.0
3772,Zamora Gas (ABD),Pacific Inland Operating Corporation,0.0,0.0


In [ ]:
# Drop zero values
wells_df = wells_df.drop(wells_df[(wells_df['OilorCondensateProduced'] == 0.0) & (wells_df['GasProduced'] == 0.0)].index)
# wells_df

# CHECK PRODUCTION TOTALS
total_gas_prod = wells_df['GasProduced'].sum()
total_oilcondens_prod = wells_df['OilorCondensateProduced'].sum()

print(f'Total gas production: {total_gas_prod}')
print(f'Total oil and condensate production: {total_oilcondens_prod}')

Total gas production: 295651129.6302209
Total oil and condensate production: 144536567.13218257


In [ ]:
# Export to Excel
wells_df.to_excel(goget_path + 'California Production by Field and Operator 2021-07-30 reformatted.xlsx', index=False)